In [66]:
import pandas as pd
import numpy as np

# Load the final 10,000-record sales dataset
sales = pd.read_csv("../data/sales_10k_clean.csv")

# Convert date column
sales["date"] = pd.to_datetime(sales["date"], errors="coerce")

# Sort data by SKU and date
sales = sales.sort_values(["sku_id", "date"]).reset_index(drop=True)

print("Records:", len(sales))
print("Columns:", sales.columns.tolist())
print("Date Range:", sales["date"].min(), "to", sales["date"].max())
print("Unique SKUs:", sales["sku_id"].nunique())

Records: 10000
Columns: ['date', 'receipt_id', 'store_id', 'sku_id', 'customer_id', 'quantity', 'unit_price', 'total_value', 'channel', 'discount_pct', 'promo_id']
Date Range: 2022-01-01 00:00:00 to 2025-12-31 00:00:00
Unique SKUs: 3482


In [67]:
weekly_sku = (
    sales
    .set_index("date")
    .groupby("sku_id")
    .resample("W")
    .agg(
        units_sold=("quantity", "sum"),
        revenue=("total_value", "sum"),
        transactions=("receipt_id", "nunique")
    )
    .reset_index()
)

print("Weekly SKU records:", len(weekly_sku))
print("Unique SKUs:", weekly_sku["sku_id"].nunique())

weekly_sku.head(10)

Weekly SKU records: 198856
Unique SKUs: 3482


,sku_id,date,units_sold,revenue,transactions
0,SKU00001,2022-07-24,3,2440.23,1
1,SKU00001,2022-07-31,0,0.00,0
2,SKU00001,2022-08-07,0,0.00,0
3,SKU00001,2022-08-14,0,0.00,0
4,SKU00001,2022-08-21,0,0.00,0
5,SKU00001,2022-08-28,0,0.00,0
6,SKU00001,2022-09-04,0,0.00,0
7,SKU00001,2022-09-11,0,0.00,0
8,SKU00001,2022-09-18,0,0.00,0
9,SKU00001,2022-09-25,0,0.00,0


In [68]:
print("Missing values:")
print(weekly_sku.isna().sum())

print("\nDuplicate SKU-Week records:")
print(
    weekly_sku.duplicated(
        subset=["sku_id", "date"]
    ).sum()
)

print("\nDate Range:")
print(weekly_sku["date"].min(), "to", weekly_sku["date"].max())

Missing values:
sku_id          0
date            0
units_sold      0
revenue         0
transactions    0
dtype: int64

Duplicate SKU-Week records:
0

Date Range:
2022-01-02 00:00:00 to 2026-01-04 00:00:00


In [69]:
# Create lag features for each SKU

weekly_sku["lag_1"] = (
    weekly_sku.groupby("sku_id")["units_sold"].shift(1)
)

weekly_sku["lag_2"] = (
    weekly_sku.groupby("sku_id")["units_sold"].shift(2)
)

weekly_sku["lag_4"] = (
    weekly_sku.groupby("sku_id")["units_sold"].shift(4)
)

print("Lag features created successfully.")

weekly_sku[
    [
        "sku_id",
        "date",
        "units_sold",
        "lag_1",
        "lag_2",
        "lag_4"
    ]
].head(15)

Lag features created successfully.


,sku_id,date,units_sold,lag_1,lag_2,lag_4
0,SKU00001,2022-07-24,3,NaN,NaN,NaN
1,SKU00001,2022-07-31,0,3.0,NaN,NaN
2,SKU00001,2022-08-07,0,0.0,3.0,NaN
3,SKU00001,2022-08-14,0,0.0,0.0,NaN
4,SKU00001,2022-08-21,0,0.0,0.0,3.0
5,SKU00001,2022-08-28,0,0.0,0.0,0.0
6,SKU00001,2022-09-04,0,0.0,0.0,0.0
7,SKU00001,2022-09-11,0,0.0,0.0,0.0
8,SKU00001,2022-09-18,0,0.0,0.0,0.0
9,SKU00001,2022-09-25,0,0.0,0.0,0.0


In [70]:
# Create rolling demand features

weekly_sku["rolling_mean_4"] = (
    weekly_sku
    .groupby("sku_id")["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(window=4).mean()
    )
)

weekly_sku["rolling_mean_8"] = (
    weekly_sku
    .groupby("sku_id")["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(window=8).mean()
    )
)

print("Rolling features created successfully.")

weekly_sku[
    [
        "sku_id",
        "date",
        "units_sold",
        "lag_1",
        "rolling_mean_4",
        "rolling_mean_8"
    ]
].head(15)

Rolling features created successfully.


,sku_id,date,units_sold,lag_1,rolling_mean_4,rolling_mean_8
0,SKU00001,2022-07-24,3,NaN,NaN,NaN
1,SKU00001,2022-07-31,0,3.0,NaN,NaN
2,SKU00001,2022-08-07,0,0.0,NaN,NaN
3,SKU00001,2022-08-14,0,0.0,NaN,NaN
4,SKU00001,2022-08-21,0,0.0,0.75,NaN
5,SKU00001,2022-08-28,0,0.0,0.00,NaN
6,SKU00001,2022-09-04,0,0.0,0.00,NaN
7,SKU00001,2022-09-11,0,0.0,0.00,NaN
8,SKU00001,2022-09-18,0,0.0,0.00,0.375
9,SKU00001,2022-09-25,0,0.0,0.00,0.000


In [71]:
# Create calendar-based features

weekly_sku["year"] = weekly_sku["date"].dt.year
weekly_sku["month"] = weekly_sku["date"].dt.month
weekly_sku["week_of_year"] = weekly_sku["date"].dt.isocalendar().week.astype(int)
weekly_sku["quarter"] = weekly_sku["date"].dt.quarter

print("Calendar features created successfully.")

weekly_sku[
    [
        "sku_id",
        "date",
        "year",
        "month",
        "week_of_year",
        "quarter"
    ]
].head(10)

Calendar features created successfully.


,sku_id,date,year,month,week_of_year,quarter
0,SKU00001,2022-07-24,2022,7,29,3
1,SKU00001,2022-07-31,2022,7,30,3
2,SKU00001,2022-08-07,2022,8,31,3
3,SKU00001,2022-08-14,2022,8,32,3
4,SKU00001,2022-08-21,2022,8,33,3
5,SKU00001,2022-08-28,2022,8,34,3
6,SKU00001,2022-09-04,2022,9,35,3
7,SKU00001,2022-09-11,2022,9,36,3
8,SKU00001,2022-09-18,2022,9,37,3
9,SKU00001,2022-09-25,2022,9,38,3


In [72]:
# Create weekly promotion features from the 10K sales data

sales["is_promotion"] = sales["promo_id"].notna()

weekly_promo = (
    sales
    .set_index("date")
    .groupby("sku_id")
    .resample("W")
    .agg(
        promo_transactions=("is_promotion", "sum"),
        avg_discount=("discount_pct", "mean")
    )
    .reset_index()
)

weekly_promo["promo_flag"] = (
    weekly_promo["promo_transactions"] > 0
).astype(int)

print("Weekly promotion features created.")

weekly_promo.head(10)

Weekly promotion features created.


,sku_id,date,promo_transactions,avg_discount,promo_flag
0,SKU00001,2022-07-24,0,0.0,0
1,SKU00001,2022-07-31,0,NaN,0
2,SKU00001,2022-08-07,0,NaN,0
3,SKU00001,2022-08-14,0,NaN,0
4,SKU00001,2022-08-21,0,NaN,0
5,SKU00001,2022-08-28,0,NaN,0
6,SKU00001,2022-09-04,0,NaN,0
7,SKU00001,2022-09-11,0,NaN,0
8,SKU00001,2022-09-18,0,NaN,0
9,SKU00001,2022-09-25,0,NaN,0


In [73]:
weekly_sku = weekly_sku.merge(
    weekly_promo[
        [
            "sku_id",
            "date",
            "promo_transactions",
            "avg_discount",
            "promo_flag"
        ]
    ],
    on=["sku_id", "date"],
    how="left"
)

weekly_sku[
    [
        "sku_id",
        "date",
        "units_sold",
        "promo_transactions",
        "avg_discount",
        "promo_flag"
    ]
].head(15)

,sku_id,date,units_sold,promo_transactions,avg_discount,promo_flag
0,SKU00001,2022-07-24,3,0,0.0,0
1,SKU00001,2022-07-31,0,0,NaN,0
2,SKU00001,2022-08-07,0,0,NaN,0
3,SKU00001,2022-08-14,0,0,NaN,0
4,SKU00001,2022-08-21,0,0,NaN,0
5,SKU00001,2022-08-28,0,0,NaN,0
6,SKU00001,2022-09-04,0,0,NaN,0
7,SKU00001,2022-09-11,0,0,NaN,0
8,SKU00001,2022-09-18,0,0,NaN,0
9,SKU00001,2022-09-25,0,0,NaN,0


In [74]:
print("===== FEATURE QUALITY CHECK =====")

print("Total records:", len(weekly_sku))
print("Total columns:", len(weekly_sku.columns))

print("\nMissing values:")
print(
    weekly_sku[
        [
            "lag_1",
            "lag_2",
            "lag_4",
            "rolling_mean_4",
            "rolling_mean_8",
            "promo_transactions",
            "avg_discount",
            "promo_flag"
        ]
    ].isna().sum()
)

print("\nDuplicate SKU-Week records:")
print(
    weekly_sku.duplicated(
        subset=["sku_id", "date"]
    ).sum()
)

===== FEATURE QUALITY CHECK =====
Total records: 198856
Total columns: 17

Missing values:
lag_1                   3482
lag_2                   5414
lag_4                   9254
rolling_mean_4          9254
rolling_mean_8         16838
promo_transactions         0
avg_discount          189515
promo_flag                 0
dtype: int64

Duplicate SKU-Week records:
0


In [75]:
feature_cols = [
    "lag_1",
    "lag_2",
    "lag_4",
    "rolling_mean_4",
    "rolling_mean_8"
]

before = len(weekly_sku)

weekly_model = weekly_sku.dropna(
    subset=feature_cols
).copy()

after = len(weekly_model)

print("Records before:", before)
print("Records after:", after)
print("Records removed:", before - after)

print("\nRemaining missing values:")
print(weekly_model[feature_cols].isna().sum())

Records before: 198856
Records after: 182018
Records removed: 16838

Remaining missing values:
lag_1             0
lag_2             0
lag_4             0
rolling_mean_4    0
rolling_mean_8    0
dtype: int64


In [76]:
# Fill promotion features for non-promotion weeks

weekly_model["promo_transactions"] = (
    weekly_model["promo_transactions"].fillna(0)
)

weekly_model["avg_discount"] = (
    weekly_model["avg_discount"].fillna(0)
)

weekly_model["promo_flag"] = (
    weekly_model["promo_flag"].fillna(0).astype(int)
)

# Sort final dataset
weekly_model = weekly_model.sort_values(
    ["sku_id", "date"]
).reset_index(drop=True)

print("===== FINAL FEATURE DATASET =====")
print("Records:", len(weekly_model))
print("Columns:", len(weekly_model.columns))
print("Unique SKUs:", weekly_model["sku_id"].nunique())

print("\nRemaining missing values:")
print(weekly_model.isna().sum())

weekly_model.head(10)

===== FINAL FEATURE DATASET =====
Records: 182018
Columns: 17
Unique SKUs: 1877

Remaining missing values:
sku_id                0
date                  0
units_sold            0
revenue               0
transactions          0
lag_1                 0
lag_2                 0
lag_4                 0
rolling_mean_4        0
rolling_mean_8        0
year                  0
month                 0
week_of_year          0
quarter               0
promo_transactions    0
avg_discount          0
promo_flag            0
dtype: int64


,sku_id,date,units_sold,revenue,transactions,lag_1,lag_2,lag_4,rolling_mean_4,rolling_mean_8,year,month,week_of_year,quarter,promo_transactions,avg_discount,promo_flag
0,SKU00001,2022-09-18,0,0.0,0,0.0,0.0,0.0,0.0,0.375,2022,9,37,3,0,0.0,0
1,SKU00001,2022-09-25,0,0.0,0,0.0,0.0,0.0,0.0,0.000,2022,9,38,3,0,0.0,0
2,SKU00001,2022-10-02,0,0.0,0,0.0,0.0,0.0,0.0,0.000,2022,10,39,4,0,0.0,0
3,SKU00001,2022-10-09,0,0.0,0,0.0,0.0,0.0,0.0,0.000,2022,10,40,4,0,0.0,0
4,SKU00001,2022-10-16,0,0.0,0,0.0,0.0,0.0,0.0,0.000,2022,10,41,4,0,0.0,0
5,SKU00001,2022-10-23,0,0.0,0,0.0,0.0,0.0,0.0,0.000,2022,10,42,4,0,0.0,0
6,SKU00001,2022-10-30,0,0.0,0,0.0,0.0,0.0,0.0,0.000,2022,10,43,4,0,0.0,0
7,SKU00001,2022-11-06,0,0.0,0,0.0,0.0,0.0,0.0,0.000,2022,11,44,4,0,0.0,0
8,SKU00001,2022-11-13,0,0.0,0,0.0,0.0,0.0,0.0,0.000,2022,11,45,4,0,0.0,0
9,SKU00001,2022-11-20,0,0.0,0,0.0,0.0,0.0,0.0,0.000,2022,11,46,4,0,0.0,0


In [77]:
# Create 52-week seasonal lag

weekly_model["lag_52"] = (
    weekly_model
    .groupby("sku_id")["units_sold"]
    .shift(52)
)

print("52-week seasonal lag created.")

weekly_model[
    [
        "sku_id",
        "date",
        "units_sold",
        "lag_52"
    ]
].head(20)

52-week seasonal lag created.


,sku_id,date,units_sold,lag_52
0,SKU00001,2022-09-18,0,NaN
1,SKU00001,2022-09-25,0,NaN
2,SKU00001,2022-10-02,0,NaN
3,SKU00001,2022-10-09,0,NaN
4,SKU00001,2022-10-16,0,NaN
5,SKU00001,2022-10-23,0,NaN
6,SKU00001,2022-10-30,0,NaN
7,SKU00001,2022-11-06,0,NaN
8,SKU00001,2022-11-13,0,NaN
9,SKU00001,2022-11-20,0,NaN


In [78]:
print("Total records:", len(weekly_model))

print(
    "Records with 52-week history:",
    weekly_model["lag_52"].notna().sum()
)

print(
    "Records without 52-week history:",
    weekly_model["lag_52"].isna().sum()
)

print(
    "52-week history availability:",
    f"{weekly_model['lag_52'].notna().mean() * 100:.2f}%"
)

Total records: 182018
Records with 52-week history: 96606
Records without 52-week history: 85412
52-week history availability: 53.07%


In [79]:
# Create Seasonal-Naive forecast
# Forecast for a week = demand from the same SKU 52 weeks earlier

baseline_data = weekly_model[
    weekly_model["lag_52"].notna()
].copy()

baseline_data["seasonal_naive_forecast"] = baseline_data["lag_52"]

print("Baseline records:", len(baseline_data))

baseline_data[
    [
        "sku_id",
        "date",
        "units_sold",
        "lag_52",
        "seasonal_naive_forecast"
    ]
].head(20)

Baseline records: 96606


,sku_id,date,units_sold,lag_52,seasonal_naive_forecast
52,SKU00001,2023-09-17,0,0.0,0.0
53,SKU00001,2023-09-24,0,0.0,0.0
54,SKU00001,2023-10-01,0,0.0,0.0
55,SKU00001,2023-10-08,0,0.0,0.0
56,SKU00001,2023-10-15,0,0.0,0.0
57,SKU00001,2023-10-22,0,0.0,0.0
58,SKU00001,2023-10-29,0,0.0,0.0
59,SKU00001,2023-11-05,0,0.0,0.0
60,SKU00001,2023-11-12,0,0.0,0.0
61,SKU00001,2023-11-19,0,0.0,0.0


In [80]:
# Calculate WAPE for Seasonal-Naive baseline

absolute_error = (
    baseline_data["units_sold"] -
    baseline_data["seasonal_naive_forecast"]
).abs()

total_actual = baseline_data["units_sold"].sum()

wape = (absolute_error.sum() / total_actual) * 100

print("===== SEASONAL-NAIVE BASELINE =====")
print("WAPE:", round(wape, 2), "%")

===== SEASONAL-NAIVE BASELINE =====
WAPE: 142.26 %


In [81]:
# Calculate forecast bias

bias = (
    baseline_data["seasonal_naive_forecast"]
    - baseline_data["units_sold"]
).mean()

print("===== SEASONAL-NAIVE BIAS =====")
print("Bias:", round(bias, 2))

===== SEASONAL-NAIVE BIAS =====
Bias: -0.03


In [82]:
print("===== ML MODEL DATA CHECK =====")

print("Records:", len(weekly_model))
print("Columns:", weekly_model.columns.tolist())

print("\nDate Range:")
print(weekly_model["date"].min(), "to", weekly_model["date"].max())

print("\nUnique SKUs:", weekly_model["sku_id"].nunique())

===== ML MODEL DATA CHECK =====
Records: 182018
Columns: ['sku_id', 'date', 'units_sold', 'revenue', 'transactions', 'lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'rolling_mean_8', 'year', 'month', 'week_of_year', 'quarter', 'promo_transactions', 'avg_discount', 'promo_flag', 'lag_52']

Date Range:
2022-02-27 00:00:00 to 2026-01-04 00:00:00

Unique SKUs: 1877


In [83]:
# ===== ML MODEL FEATURE PREPARATION =====

target = "units_sold"

features = [
    "lag_1",
    "lag_2",
    "lag_4",
    "rolling_mean_4",
    "rolling_mean_8",
    "year",
    "month",
    "week_of_year",
    "quarter",
    "promo_transactions",
    "avg_discount",
    "promo_flag",
    "lag_52"
]

model_data = weekly_model[
    ["sku_id", "date", target] + features
].copy()

print("===== MODEL DATA =====")
print("Records:", len(model_data))
print("Features:", len(features))
print("Target:", target)

print("\nMissing values:")
print(model_data[features + [target]].isna().sum().sum())

===== MODEL DATA =====
Records: 182018
Features: 13
Target: units_sold

Missing values:
85412


In [84]:
# ===== PREPARE FINAL ML DATA =====

ml_data = model_data.dropna(
    subset=features + [target]
).copy()

# Date ke according sort
ml_data = ml_data.sort_values(
    ["date", "sku_id"]
).reset_index(drop=True)

print("===== FINAL ML DATA =====")
print("Records:", len(ml_data))
print("Unique SKUs:", ml_data["sku_id"].nunique())
print("Date Range:", ml_data["date"].min(), "to", ml_data["date"].max())

print("\nMissing values:")
print(ml_data[features + [target]].isna().sum().sum())

===== FINAL ML DATA =====
Records: 96606
Unique SKUs: 1388
Date Range: 2023-02-26 00:00:00 to 2026-01-04 00:00:00

Missing values:
0


In [85]:
# ===== TIME-BASED TRAIN / TEST SPLIT =====

split_date = ml_data["date"].quantile(0.80)

train_data = ml_data[
    ml_data["date"] <= split_date
].copy()

test_data = ml_data[
    ml_data["date"] > split_date
].copy()

print("===== TIME-BASED SPLIT =====")

print("Split Date:", split_date)

print("\nTRAIN DATA")
print("Records:", len(train_data))
print("Date Range:", train_data["date"].min(), "to", train_data["date"].max())

print("\nTEST DATA")
print("Records:", len(test_data))
print("Date Range:", test_data["date"].min(), "to", test_data["date"].max())

===== TIME-BASED SPLIT =====
Split Date: 2025-03-30 00:00:00

TRAIN DATA
Records: 77533
Date Range: 2023-02-26 00:00:00 to 2025-03-30 00:00:00

TEST DATA
Records: 19073
Date Range: 2025-04-06 00:00:00 to 2026-01-04 00:00:00


In [86]:
# ===== ML MODEL SETUP =====

%pip install -q scikit-learn

from sklearn.ensemble import RandomForestRegressor

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("===== ML MODEL SETUP =====")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

Note: you may need to restart the kernel to use updated packages.
===== ML MODEL SETUP =====
X_train: (77533, 13)
y_train: (77533,)
X_test : (19073, 13)
y_test : (19073,)


In [87]:
# ===== RANDOM FOREST MODEL TRAINING =====

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    max_depth=20,
    min_samples_leaf=2
)

print("Training Random Forest model...")

rf_model.fit(X_train, y_train)

print("===== MODEL TRAINING COMPLETE =====")
print("Model:", rf_model)

Training Random Forest model...
===== MODEL TRAINING COMPLETE =====
Model: RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=200,
                      n_jobs=-1, random_state=42)


In [88]:
# ===== RANDOM FOREST PREDICTIONS =====

rf_predictions = rf_model.predict(X_test)

test_results = test_data[
    ["sku_id", "date", "units_sold"]
].copy()

test_results["rf_forecast"] = rf_predictions

print("===== PREDICTIONS COMPLETE =====")
print("Prediction records:", len(test_results))

print("\nSample predictions:")
print(
    test_results[
        ["sku_id", "date", "units_sold", "rf_forecast"]
    ].head(10)
)

===== PREDICTIONS COMPLETE =====
Prediction records: 19073

Sample predictions:
         sku_id       date  units_sold  rf_forecast
77533  SKU00001 2025-04-06           0     0.053678
77534  SKU00002 2025-04-06           0     0.053678
77535  SKU00010 2025-04-06           0     0.053678
77536  SKU00014 2025-04-06           0     0.053678
77537  SKU00016 2025-04-06           0     0.053678
77538  SKU00020 2025-04-06           0     0.053678
77539  SKU00024 2025-04-06           0     0.053678
77540  SKU00030 2025-04-06           0     0.053678
77541  SKU00036 2025-04-06           0     0.009112
77542  SKU00038 2025-04-06           0     0.053678


In [89]:
# ===== RANDOM FOREST WAPE =====

absolute_error = (
    test_results["units_sold"] -
    test_results["rf_forecast"]
).abs()

total_actual = test_results["units_sold"].sum()

rf_wape = (
    absolute_error.sum() / total_actual
) * 100

print("===== RANDOM FOREST PERFORMANCE =====")
print("WAPE:", round(rf_wape, 2), "%")

===== RANDOM FOREST PERFORMANCE =====
WAPE: 132.73 %


In [90]:
# ===== RANDOM FOREST BIAS =====

rf_bias = (
    test_results["rf_forecast"] -
    test_results["units_sold"]
).mean()

print("===== RANDOM FOREST BIAS =====")
print("Bias:", round(rf_bias, 2))

===== RANDOM FOREST BIAS =====
Bias: -0.07


In [91]:
# ===== ROLLING-ORIGIN BACKTEST: DATE CHECK =====

unique_dates = sorted(ml_data["date"].unique())

print("===== BACKTEST DATE RANGE =====")
print("Total weekly dates:", len(unique_dates))

print("\nFirst 5 dates:")
print(unique_dates[:5])

print("\nLast 5 dates:")
print(unique_dates[-5:])

===== BACKTEST DATE RANGE =====
Total weekly dates: 150

First 5 dates:
[Timestamp('2023-02-26 00:00:00'), Timestamp('2023-03-05 00:00:00'), Timestamp('2023-03-12 00:00:00'), Timestamp('2023-03-19 00:00:00'), Timestamp('2023-03-26 00:00:00')]

Last 5 dates:
[Timestamp('2025-12-07 00:00:00'), Timestamp('2025-12-14 00:00:00'), Timestamp('2025-12-21 00:00:00'), Timestamp('2025-12-28 00:00:00'), Timestamp('2026-01-04 00:00:00')]


In [92]:
# ===== ROLLING-ORIGIN BACKTEST WINDOWS =====

backtest_windows = [
    {
        "train_end": "2024-12-29",
        "test_start": "2025-01-05",
        "test_end": "2025-01-26"
    },
    {
        "train_end": "2025-03-30",
        "test_start": "2025-04-06",
        "test_end": "2025-04-27"
    },
    {
        "train_end": "2025-06-29",
        "test_start": "2025-07-06",
        "test_end": "2025-07-27"
    }
]

print("===== ROLLING BACKTEST WINDOWS =====")

for i, window in enumerate(backtest_windows, start=1):
    print(f"\nWindow {i}")
    print("Train End :", window["train_end"])
    print("Test      :", window["test_start"], "to", window["test_end"])

===== ROLLING BACKTEST WINDOWS =====

Window 1
Train End : 2024-12-29
Test      : 2025-01-05 to 2025-01-26

Window 2
Train End : 2025-03-30
Test      : 2025-04-06 to 2025-04-27

Window 3
Train End : 2025-06-29
Test      : 2025-07-06 to 2025-07-27


In [93]:
# ===== ROLLING-ORIGIN BACKTEST =====

from sklearn.ensemble import RandomForestRegressor

backtest_results = []

for i, window in enumerate(backtest_windows, start=1):

    train_end = pd.to_datetime(window["train_end"])
    test_start = pd.to_datetime(window["test_start"])
    test_end = pd.to_datetime(window["test_end"])

    # Time-based train/test data
    bt_train = ml_data[
        ml_data["date"] <= train_end
    ].copy()

    bt_test = ml_data[
        (ml_data["date"] >= test_start) &
        (ml_data["date"] <= test_end)
    ].copy()

    # Features and target
    X_bt_train = bt_train[features]
    y_bt_train = bt_train[target]

    X_bt_test = bt_test[features]
    y_bt_test = bt_test[target]

    # Model
    bt_model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        max_depth=20,
        min_samples_leaf=2
    )

    print(f"\nRunning Window {i}...")
    print("Train records:", len(bt_train))
    print("Test records :", len(bt_test))

    # Train
    bt_model.fit(X_bt_train, y_bt_train)

    # Predict
    bt_pred = bt_model.predict(X_bt_test)

    # WAPE
    bt_absolute_error = (
        y_bt_test - bt_pred
    ).abs()

    bt_total_actual = y_bt_test.sum()

    bt_wape = (
        bt_absolute_error.sum() / bt_total_actual
    ) * 100

    # Bias
    bt_bias = (
        bt_pred - y_bt_test
    ).mean()

    backtest_results.append({
        "window": i,
        "train_end": train_end,
        "test_start": test_start,
        "test_end": test_end,
        "train_records": len(bt_train),
        "test_records": len(bt_test),
        "wape": bt_wape,
        "bias": bt_bias
    })

print("\n===== ROLLING BACKTEST COMPLETE =====")

backtest_df = pd.DataFrame(backtest_results)

print(
    backtest_df[
        [
            "window",
            "train_end",
            "test_start",
            "test_end",
            "wape",
            "bias"
        ]
    ]
)


Running Window 1...
Train records: 66884
Test records : 3426

Running Window 2...
Train records: 77533
Test records : 2984

Running Window 3...
Train records: 86561
Test records : 2311

===== ROLLING BACKTEST COMPLETE =====
   window  train_end test_start   test_end        wape      bias
0       1 2024-12-29 2025-01-05 2025-01-26  133.449269 -0.021991
1       2 2025-03-30 2025-04-06 2025-04-27  161.001671 -0.010843
2       3 2025-06-29 2025-07-06 2025-07-27  152.156452 -0.035169


In [94]:
# ===== OVERALL ROLLING BACKTEST PERFORMANCE =====

avg_wape = backtest_df["wape"].mean()
avg_bias = backtest_df["bias"].mean()

print("===== OVERALL BACKTEST PERFORMANCE =====")
print("Average WAPE:", round(avg_wape, 2), "%")
print("Average Bias:", round(avg_bias, 4))

print("\nBaseline WAPE:", 142.26, "%")
print("Single Test WAPE:", 132.73, "%")

improvement = 142.26 - avg_wape

print(
    "Baseline vs Backtest WAPE improvement:",
    round(improvement, 2),
    "percentage points"
)

===== OVERALL BACKTEST PERFORMANCE =====
Average WAPE: 148.87 %
Average Bias: -0.0227

Baseline WAPE: 142.26 %
Single Test WAPE: 132.73 %
Baseline vs Backtest WAPE improvement: -6.61 percentage points


In [95]:
# ===== FEATURE IMPORTANCE =====

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("===== FEATURE IMPORTANCE =====")
print(feature_importance)

===== FEATURE IMPORTANCE =====
               feature  importance
0       rolling_mean_8    0.241690
1   promo_transactions    0.197515
2         avg_discount    0.151597
3           promo_flag    0.123785
4         week_of_year    0.082346
5       rolling_mean_4    0.048514
6               lag_52    0.037094
7                lag_4    0.031198
8                lag_1    0.027253
9                lag_2    0.023126
10               month    0.016419
11                year    0.016060
12             quarter    0.003401


In [96]:
# ===== SEASONAL-NAIVE ON SAME BACKTEST WINDOWS =====

baseline_backtest_results = []

for i, window in enumerate(backtest_windows, start=1):

    test_start = pd.to_datetime(window["test_start"])
    test_end = pd.to_datetime(window["test_end"])

    bt_test = ml_data[
        (ml_data["date"] >= test_start) &
        (ml_data["date"] <= test_end)
    ].copy()

    # Seasonal-naive forecast
    bt_test["naive_forecast"] = bt_test["lag_52"]

    # WAPE
    naive_error = (
        bt_test["units_sold"] -
        bt_test["naive_forecast"]
    ).abs()

    naive_wape = (
        naive_error.sum() /
        bt_test["units_sold"].sum()
    ) * 100

    # Bias
    naive_bias = (
        bt_test["naive_forecast"] -
        bt_test["units_sold"]
    ).mean()

    baseline_backtest_results.append({
        "window": i,
        "wape": naive_wape,
        "bias": naive_bias
    })

baseline_backtest_df = pd.DataFrame(
    baseline_backtest_results
)

print("===== SEASONAL-NAIVE ROLLING BACKTEST =====")
print(baseline_backtest_df)

print("\nAverage WAPE:",
      round(baseline_backtest_df["wape"].mean(), 2), "%")

print("Average Bias:",
      round(baseline_backtest_df["bias"].mean(), 4))

===== SEASONAL-NAIVE ROLLING BACKTEST =====
   window        wape      bias
0       1  147.979798 -0.023059
1       2  152.014652 -0.024464
2       3  132.824427 -0.065772

Average WAPE: 144.27 %
Average Bias: -0.0378


In [97]:
# ===== GRADIENT BOOSTING MODEL SETUP =====

from sklearn.ensemble import GradientBoostingRegressor

gbr_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

print("===== GRADIENT BOOSTING MODEL =====")
print(gbr_model)

===== GRADIENT BOOSTING MODEL =====
GradientBoostingRegressor(learning_rate=0.05, max_depth=5, min_samples_leaf=5,
                          n_estimators=200, random_state=42)


In [98]:
# ===== GRADIENT BOOSTING MODEL TRAINING =====

print("Training Gradient Boosting model...")

gbr_model.fit(X_train, y_train)

print("===== MODEL TRAINING COMPLETE =====")
print("Model:", gbr_model)

Training Gradient Boosting model...
===== MODEL TRAINING COMPLETE =====
Model: GradientBoostingRegressor(learning_rate=0.05, max_depth=5, min_samples_leaf=5,
                          n_estimators=200, random_state=42)


In [99]:
# ===== GRADIENT BOOSTING PREDICTIONS =====

gbr_predictions = gbr_model.predict(X_test)

gbr_test_results = test_data[
    ["sku_id", "date", "units_sold"]
].copy()

gbr_test_results["gbr_forecast"] = gbr_predictions

print("===== GRADIENT BOOSTING PREDICTIONS =====")
print("Prediction records:", len(gbr_test_results))

print("\nSample predictions:")
print(
    gbr_test_results[
        ["sku_id", "date", "units_sold", "gbr_forecast"]
    ].head(10)
)

===== GRADIENT BOOSTING PREDICTIONS =====
Prediction records: 19073

Sample predictions:
         sku_id       date  units_sold  gbr_forecast
77533  SKU00001 2025-04-06           0      0.052742
77534  SKU00002 2025-04-06           0      0.052742
77535  SKU00010 2025-04-06           0      0.052742
77536  SKU00014 2025-04-06           0      0.052742
77537  SKU00016 2025-04-06           0      0.052742
77538  SKU00020 2025-04-06           0      0.052742
77539  SKU00024 2025-04-06           0      0.052742
77540  SKU00030 2025-04-06           0      0.052742
77541  SKU00036 2025-04-06           0      0.065471
77542  SKU00038 2025-04-06           0      0.052742


In [100]:
# ===== GRADIENT BOOSTING WAPE =====

gbr_absolute_error = (
    gbr_test_results["units_sold"] -
    gbr_test_results["gbr_forecast"]
).abs()

gbr_total_actual = gbr_test_results["units_sold"].sum()

gbr_wape = (
    gbr_absolute_error.sum() /
    gbr_total_actual
) * 100

print("===== GRADIENT BOOSTING PERFORMANCE =====")
print("WAPE:", round(gbr_wape, 2), "%")

===== GRADIENT BOOSTING PERFORMANCE =====
WAPE: 132.22 %


In [101]:
# ===== GRADIENT BOOSTING BIAS =====

gbr_bias = (
    gbr_test_results["gbr_forecast"] -
    gbr_test_results["units_sold"]
).mean()

print("===== GRADIENT BOOSTING BIAS =====")
print("Bias:", round(gbr_bias, 2))

===== GRADIENT BOOSTING BIAS =====
Bias: -0.07


In [102]:
# ===== GRADIENT BOOSTING ROLLING BACKTEST =====

gbr_rolling_results = []

rolling_windows = [
    ("2024-12-29", "2025-01-05", "2025-01-26"),
    ("2025-03-30", "2025-04-06", "2025-04-27"),
    ("2025-06-29", "2025-07-06", "2025-07-27")
]

for train_end, test_start, test_end in rolling_windows:

    train_window = ml_data[
        ml_data["date"] <= train_end
    ].copy()

    test_window = ml_data[
        (ml_data["date"] >= test_start) &
        (ml_data["date"] <= test_end)
    ].copy()

    X_train_window = train_window[features]
    y_train_window = train_window[target]

    X_test_window = test_window[features]
    y_test_window = test_window[target]

    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        min_samples_leaf=5,
        random_state=42
    )

    model.fit(X_train_window, y_train_window)

    predictions = model.predict(X_test_window)

    absolute_error = (
        y_test_window - predictions
    ).abs()

    wape = (
        absolute_error.sum() /
        y_test_window.sum()
    ) * 100

    bias = (
        predictions - y_test_window
    ).mean()

    gbr_rolling_results.append({
        "train_end": train_end,
        "test_start": test_start,
        "test_end": test_end,
        "train_records": len(train_window),
        "test_records": len(test_window),
        "wape": wape,
        "bias": bias
    })

gbr_rolling_results = pd.DataFrame(gbr_rolling_results)

print("===== GRADIENT BOOSTING ROLLING BACKTEST =====")
print(gbr_rolling_results)

===== GRADIENT BOOSTING ROLLING BACKTEST =====
    train_end  test_start    test_end  train_records  test_records  \
0  2024-12-29  2025-01-05  2025-01-26          66884          3426   
1  2025-03-30  2025-04-06  2025-04-27          77533          2984   
2  2025-06-29  2025-07-06  2025-07-27          86561          2311   

         wape      bias  
0  133.513781 -0.022239  
1  156.308664 -0.018549  
2  162.186208 -0.024467  


In [103]:
# ============================================================
# GRADIENT BOOSTING - EVALUATION METRICS
# MAE, MSE, RMSE, R2, WAPE and Bias
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ------------------------------------------------------------
# STEP 1: Define features and target
# ------------------------------------------------------------

features = [
    "lag_1",
    "lag_2",
    "lag_4",
    "rolling_mean_4",
    "rolling_mean_8",
    "lag_52",
    "year",
    "month",
    "week_of_year",
    "quarter",
    "promo_transactions",
    "avg_discount",
    "promo_flag"
]

target = "units_sold"


# ------------------------------------------------------------
# STEP 2: Define rolling test windows
# ------------------------------------------------------------

rolling_windows = [
    ("2024-12-29", "2025-01-05", "2025-01-26"),
    ("2025-03-30", "2025-04-06", "2025-04-27"),
    ("2025-06-29", "2025-07-06", "2025-07-27")
]


# ------------------------------------------------------------
# STEP 3: Create empty list for results
# ------------------------------------------------------------

evaluation_results = []


# ------------------------------------------------------------
# STEP 4: Run model for each rolling window
# ------------------------------------------------------------

for train_end, test_start, test_end in rolling_windows:

    # Training data
    train_data = ml_data[
        ml_data["date"] <= train_end
    ].copy()

    # Testing data
    test_data = ml_data[
        (ml_data["date"] >= test_start) &
        (ml_data["date"] <= test_end)
    ].copy()


    # --------------------------------------------------------
    # STEP 5: Select X and y
    # --------------------------------------------------------

    X_train = train_data[features]
    y_train = train_data[target]

    X_test = test_data[features]
    y_test = test_data[target]


    # --------------------------------------------------------
    # STEP 6: Create Gradient Boosting model
    # --------------------------------------------------------

    gbr_model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        min_samples_leaf=5,
        random_state=42
    )


    # --------------------------------------------------------
    # STEP 7: Train model
    # --------------------------------------------------------

    gbr_model.fit(X_train, y_train)


    # --------------------------------------------------------
    # STEP 8: Generate predictions
    # --------------------------------------------------------

    y_pred = gbr_model.predict(X_test)


    # --------------------------------------------------------
    # STEP 9: Calculate MAE
    # --------------------------------------------------------

    mae = mean_absolute_error(
        y_test,
        y_pred
    )


    # --------------------------------------------------------
    # STEP 10: Calculate MSE
    # --------------------------------------------------------

    mse = mean_squared_error(
        y_test,
        y_pred
    )


    # --------------------------------------------------------
    # STEP 11: Calculate RMSE
    # --------------------------------------------------------

    rmse = np.sqrt(mse)


    # --------------------------------------------------------
    # STEP 12: Calculate R2 Score
    # --------------------------------------------------------

    r2 = r2_score(
        y_test,
        y_pred
    )


    # --------------------------------------------------------
    # STEP 13: Calculate WAPE
    # --------------------------------------------------------

    total_actual = y_test.sum()

    if total_actual != 0:

        wape = (
            np.abs(y_test - y_pred).sum()
            / total_actual
        ) * 100

    else:

        wape = np.nan


    # --------------------------------------------------------
    # STEP 14: Calculate Bias
    # --------------------------------------------------------

    bias = (
        y_pred - y_test
    ).mean()


    # --------------------------------------------------------
    # STEP 15: Save results
    # --------------------------------------------------------

    evaluation_results.append({

        "train_end": train_end,

        "test_start": test_start,

        "test_end": test_end,

        "MAE": mae,

        "MSE": mse,

        "RMSE": rmse,

        "R2": r2,

        "WAPE": wape,

        "Bias": bias
    })


# ------------------------------------------------------------
# STEP 16: Convert results into DataFrame
# ------------------------------------------------------------

gbr_evaluation = pd.DataFrame(
    evaluation_results
)


# ------------------------------------------------------------
# STEP 17: Display results
# ------------------------------------------------------------

print("================================================")
print("GRADIENT BOOSTING EVALUATION METRICS")
print("================================================")

display(
    gbr_evaluation.round(4)
)

GRADIENT BOOSTING EVALUATION METRICS


,train_end,test_start,test_end,MAE,MSE,RMSE,R2,WAPE,Bias
0,2024-12-29,2025-01-05,2025-01-26,0.0767,0.1245,0.3528,0.2474,132.7529,-0.0227
1,2025-03-30,2025-04-06,2025-04-27,0.1430,0.2340,0.4837,0.0222,156.3475,-0.0185
2,2025-06-29,2025-07-06,2025-07-27,0.1839,0.3054,0.5526,0.0741,162.1861,-0.0245


In [104]:
# ============================================================
# GRADIENT BOOSTING - FINAL REGRESSION METRICS
# ============================================================

gbr_final_metrics = gbr_evaluation[
    ["MAE", "MSE", "RMSE", "R2"]
].mean().to_frame().T

gbr_final_metrics.insert(
    0,
    "Model",
    "Gradient Boosting"
)

print("==============================================")
print("GRADIENT BOOSTING - FINAL REGRESSION METRICS")
print("==============================================")

display(gbr_final_metrics.round(4))

GRADIENT BOOSTING - FINAL REGRESSION METRICS


,Model,MAE,MSE,RMSE,R2
0,Gradient Boosting,0.1345,0.2213,0.463,0.1146


In [105]:
# ===== FINAL MODEL COMPARISON =====

comparison = pd.DataFrame({
    "Model": [
        "Seasonal-Naive",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Average_Rolling_WAPE": [
        144.27,
        148.87,
        150.67
    ],
    "Average_Rolling_Bias": [
        -0.0378,
        -0.0227,
        -0.0217
    ]
})

comparison["WAPE_Rank"] = (
    comparison["Average_Rolling_WAPE"]
    .rank(method="min")
    .astype(int)
)

comparison = comparison.sort_values(
    "WAPE_Rank"
).reset_index(drop=True)

print("===== FINAL MODEL COMPARISON =====")
print(comparison)

===== FINAL MODEL COMPARISON =====
               Model  Average_Rolling_WAPE  Average_Rolling_Bias  WAPE_Rank
0     Seasonal-Naive                144.27               -0.0378          1
1      Random Forest                148.87               -0.0227          2
2  Gradient Boosting                150.67               -0.0217          3


In [106]:
print("RF prediction first 10:")
print(rf_pred_final[:10])

print("\nGBR prediction first 10:")
print(gbr_pred_final[:10])

print("\nAre RF and GBR predictions exactly same?")
print(np.array_equal(rf_pred_final, gbr_pred_final))

print("\nMaximum absolute difference:")
print(np.max(np.abs(
    np.asarray(rf_pred_final) - np.asarray(gbr_pred_final)
)))

RF prediction first 10:
[0.05367832 0.05367832 0.05367832 0.05367832 0.05367832 0.05367832
 0.05367832 0.05367832 0.00911207 0.05367832]

GBR prediction first 10:
[0.05274214 0.05274214 0.05274214 0.05274214 0.05274214 0.05274214
 0.05274214 0.05274214 0.06547102 0.05274214]

Are RF and GBR predictions exactly same?
False

Maximum absolute difference:
3.161428129196918


In [107]:
print("test_results columns:")
print(test_results.columns.tolist())

print("\nAvailable variables containing 'season':")
print([x for x in globals() if 'season' in x.lower()])

print("\nAvailable variables containing 'naive':")
print([x for x in globals() if 'naive' in x.lower()])

test_results columns:
['sku_id', 'date', 'units_sold', 'rf_forecast']

Available variables containing 'season':
['seasonal_naive_pred_final', 'seasonal_naive_pred', 'seasonal_naive_metrics', 'seasonal_pred', 'seasonal_metrics', 'seasonal_pred_array']

Available variables containing 'naive':
['naive_error', 'naive_wape', 'naive_bias', 'seasonal_naive_pred_final', 'seasonal_naive_pred', 'seasonal_naive_metrics']


In [108]:
# ===== FINAL TEST SET CHECK =====

print("Random Forest test records:", len(test_results))
print("Gradient Boosting test records:", len(gbr_test_results))

print("\nRandom Forest date range:")
print(test_results["date"].min(), "to", test_results["date"].max())

print("\nGradient Boosting date range:")
print(gbr_test_results["date"].min(), "to", gbr_test_results["date"].max())

print("\nRF columns:")
print(test_results.columns.tolist())

print("\nGBR columns:")
print(gbr_test_results.columns.tolist())

Random Forest test records: 19073
Gradient Boosting test records: 19073

Random Forest date range:
2025-04-06 00:00:00 to 2026-01-04 00:00:00

Gradient Boosting date range:
2025-04-06 00:00:00 to 2026-01-04 00:00:00

RF columns:
['sku_id', 'date', 'units_sold', 'rf_forecast']

GBR columns:
['sku_id', 'date', 'units_sold', 'gbr_forecast']


In [109]:
# ===== ACTUAL VALUES ALIGNMENT CHECK =====

rf_actual = test_results["units_sold"].values
gbr_actual = gbr_test_results["units_sold"].values

print("RF actual records:", len(rf_actual))
print("GBR actual records:", len(gbr_actual))

print("\nAre actual values exactly same?")
print(np.array_equal(rf_actual, gbr_actual))

print("\nAre SKU IDs exactly same?")
print(
    np.array_equal(
        test_results["sku_id"].values,
        gbr_test_results["sku_id"].values
    )
)

print("\nAre dates exactly same?")
print(
    np.array_equal(
        test_results["date"].values,
        gbr_test_results["date"].values
    )
)

RF actual records: 19073
GBR actual records: 19073

Are actual values exactly same?
True

Are SKU IDs exactly same?
True

Are dates exactly same?
True


In [110]:
# ===== SEASONAL NAIVE CHECK =====

seasonal_pred_array = np.asarray(seasonal_naive_pred)

print("Seasonal Naive records:", len(seasonal_pred_array))

print("\nSeasonal Naive shape:")
print(seasonal_pred_array.shape)

print("\nFirst 10 Seasonal Naive predictions:")
print(seasonal_pred_array[:10])

Seasonal Naive records: 2311

Seasonal Naive shape:
(2311,)

First 10 Seasonal Naive predictions:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [111]:
print("test_results records:", len(test_results))
print("test_results columns:")
print(test_results.columns.tolist())

test_results records: 19073
test_results columns:
['sku_id', 'date', 'units_sold', 'rf_forecast']


In [112]:
print("X_test records:", len(X_test))
print("X_test columns:")
print(X_test.columns.tolist())

print("\nFirst 5 lag_52 values:")
print(X_test["lag_52"].head())

X_test records: 2311
X_test columns:
['lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'rolling_mean_8', 'lag_52', 'year', 'month', 'week_of_year', 'quarter', 'promo_transactions', 'avg_discount', 'promo_flag']

First 5 lag_52 values:
86561    0.0
86562    0.0
86563    0.0
86564    0.0
86565    0.0
Name: lag_52, dtype: float64


In [113]:
# ===== RECOVER ORIGINAL FINAL TEST DATA =====

split_date_final = ml_data["date"].quantile(0.80)

final_test_data = ml_data[
    ml_data["date"] > split_date_final
].copy()

print("===== FINAL TEST DATA RECOVERED =====")
print("Split Date:", split_date_final)
print("Final Test Records:", len(final_test_data))
print(
    "Date Range:",
    final_test_data["date"].min(),
    "to",
    final_test_data["date"].max()
)

print("\nRequired columns available:")
print(
    final_test_data[
        ["sku_id", "date", "units_sold", "lag_52"]
    ].head()
)

===== FINAL TEST DATA RECOVERED =====
Split Date: 2025-03-30 00:00:00
Final Test Records: 19073
Date Range: 2025-04-06 00:00:00 to 2026-01-04 00:00:00

Required columns available:
         sku_id       date  units_sold  lag_52
77533  SKU00001 2025-04-06           0     0.0
77534  SKU00002 2025-04-06           0     0.0
77535  SKU00010 2025-04-06           0     0.0
77536  SKU00014 2025-04-06           0     0.0
77537  SKU00016 2025-04-06           0     0.0


In [114]:
# ===== FINAL SEASONAL NAIVE PREDICTIONS =====

seasonal_naive_pred_final = final_test_data["lag_52"].to_numpy()

print("===== FINAL SEASONAL NAIVE PREDICTIONS =====")
print("Records:", len(seasonal_naive_pred_final))
print("Shape:", seasonal_naive_pred_final.shape)

print("\nFirst 10 predictions:")
print(seasonal_naive_pred_final[:10])

===== FINAL SEASONAL NAIVE PREDICTIONS =====
Records: 19073
Shape: (19073,)

First 10 predictions:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [115]:
# ===== FINAL MODEL COMPARISON =====

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def calculate_final_metrics(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    wape = (
        np.sum(np.abs(y_true - y_pred))
        / np.sum(np.abs(y_true))
    ) * 100

    bias = np.mean(y_pred - y_true)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R²": r2,
        "WAPE (%)": wape,
        "Bias": bias
    }


# ===== ACTUAL VALUES =====

y_true = test_results["units_sold"].to_numpy()

# ===== MODEL PREDICTIONS =====

rf_pred_final = test_results["rf_forecast"].to_numpy()

gbr_pred_final = gbr_test_results["gbr_forecast"].to_numpy()

sn_pred_final = seasonal_naive_pred_final


# ===== LENGTH CHECK =====

print("===== RECORD COUNT CHECK =====")

print("Actual:", len(y_true))
print("Seasonal Naive:", len(sn_pred_final))
print("Random Forest:", len(rf_pred_final))
print("Gradient Boosting:", len(gbr_pred_final))


# ===== CALCULATE METRICS =====

seasonal_naive_metrics = calculate_final_metrics(
    y_true,
    sn_pred_final
)

rf_metrics = calculate_final_metrics(
    y_true,
    rf_pred_final
)

gbr_metrics = calculate_final_metrics(
    y_true,
    gbr_pred_final
)


# ===== FINAL COMPARISON TABLE =====

final_model_comparison = pd.DataFrame(
    [
        seasonal_naive_metrics,
        rf_metrics,
        gbr_metrics
    ],
    index=[
        "Seasonal Naive",
        "Random Forest",
        "Gradient Boosting"
    ]
)


print("\n===== FINAL MODEL COMPARISON =====")
print(
    final_model_comparison.round(4)
)

===== RECORD COUNT CHECK =====
Actual: 19073
Seasonal Naive: 19073
Random Forest: 19073
Gradient Boosting: 19073

===== FINAL MODEL COMPARISON =====
                      MAE     MSE    RMSE      R²  WAPE (%)    Bias
Seasonal Naive     0.1915  0.5568  0.7462 -0.2891  125.3690 -0.0884
Random Forest      0.2027  0.3831  0.6190  0.1130  132.7293 -0.0652
Gradient Boosting  0.2019  0.3839  0.6196  0.1112  132.2248 -0.0674


In [116]:
# ===== FINAL FORECAST OUTPUT =====

final_forecast_output = test_results[
    ["sku_id", "date", "units_sold", "rf_forecast"]
].copy()

# Add Gradient Boosting forecast
final_forecast_output["gbr_forecast"] = (
    gbr_test_results["gbr_forecast"].to_numpy()
)

# Add Seasonal Naive forecast
final_forecast_output["seasonal_naive_forecast"] = (
    seasonal_naive_pred_final
)

print("===== FINAL FORECAST OUTPUT =====")

print("Records:", len(final_forecast_output))
print(
    "Columns:",
    final_forecast_output.columns.tolist()
)

print("\nFirst 10 records:")
print(final_forecast_output.head(10))

===== FINAL FORECAST OUTPUT =====
Records: 19073
Columns: ['sku_id', 'date', 'units_sold', 'rf_forecast', 'gbr_forecast', 'seasonal_naive_forecast']

First 10 records:
         sku_id       date  units_sold  rf_forecast  gbr_forecast  \
77533  SKU00001 2025-04-06           0     0.053678      0.052742   
77534  SKU00002 2025-04-06           0     0.053678      0.052742   
77535  SKU00010 2025-04-06           0     0.053678      0.052742   
77536  SKU00014 2025-04-06           0     0.053678      0.052742   
77537  SKU00016 2025-04-06           0     0.053678      0.052742   
77538  SKU00020 2025-04-06           0     0.053678      0.052742   
77539  SKU00024 2025-04-06           0     0.053678      0.052742   
77540  SKU00030 2025-04-06           0     0.053678      0.052742   
77541  SKU00036 2025-04-06           0     0.009112      0.065471   
77542  SKU00038 2025-04-06           0     0.053678      0.052742   

       seasonal_naive_forecast  
77533                      0.0  
77534 

In [117]:
# ===== SAVE FINAL FORECAST OUTPUT =====

forecast_output_path = "final_forecast_output.csv"

final_forecast_output.to_csv(
    forecast_output_path,
    index=False
)

print("===== FORECAST OUTPUT SAVED =====")
print("File:", forecast_output_path)
print("Records:", len(final_forecast_output))

===== FORECAST OUTPUT SAVED =====
File: final_forecast_output.csv
Records: 19073
